<a href="https://colab.research.google.com/github/Akisn277/Data-Science-Sheets/blob/main/Data_Science_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Section A: Data Cleaning

 1. Data Validation & Correction


In [ ]:
import pandas as pd
import numpy as np

#sample dataframe
df = pd.DataFrame({
    "Rating": [5, 4, 0, 6, None, 3]
})

invalid_ratings = df[(df["Rating"] < 1) | (df["Rating"] > 5)]
print("Invalid Ratings:\n", invalid_ratings)

df.loc[(df["Rating"] < 1) | (df["Rating"] > 5), "Rating"] = np.nan

df["Rating"].fillna(df["Rating"].mean(), inplace=True)

print("\nCleaned Ratings:\n", df)

Invalid Ratings:
    Rating
2     0.0
3     6.0

Cleaned Ratings:
    Rating
0     5.0
1     4.0
2     4.0
3     4.0
4     4.0
5     3.0


/tmp/ipykernel_3409/90684412.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Rating"].fillna(df["Rating"].mean(), inplace=True)


```
1. Detect invalid ratings (0, 6)?
Ans:
Condition check: Rating < 1 or Rating > 5
Filter using boolean indexing

2. Remove or correct values?
Ans:
Few invalid → drop rows
Many invalid → replace with mean/median or NaN

3. Validation rules defined how?
Ans:
Based on domain constraint
Valid range: 1 ≤ Rating ≤ 5

4. Handle missing values?
Ans:
Fill with mean/median or drop if minimal data loss
```

2. Text Preprocessing

In [ ]:
import re
#sample df
df = pd.DataFrame({
    "Review_Text": [
        "<p>Great course! 😊</p>",
        "Bad!!! 😡 #waste",
        "<div>Okay-ish 🤔</div>"
    ]
})

def clean_text(text):
    text = re.sub(r"<.*?>", "", text)  #remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  #remove special chars & emojis
    text = text.lower()  #convert to lowercase
    return text

df["Cleaned_Text"] = df["Review_Text"].apply(clean_text)
print(df)

              Review_Text   Cleaned_Text
0  <p>Great course! 😊</p>  great course 
1         Bad!!! 😡 #waste     bad  waste
2   <div>Okay-ish 🤔</div>       okayish 


```
2. Why is text preprocessing important?
Ans:
Removes noise from data
Ensures consistency in text
Improves accuracy of analysis / ML models

3. Handling Relative Dates

In [ ]:
from datetime import datetime, timedelta

df = pd.DataFrame({
    "Review_Date": ["Yesterday", "2 days ago", "March 5th, 2023"]
})

def convert_date(text):
    today = datetime.today()

    if text.lower() == "yesterday":
        return today - timedelta(days=1)
    elif "days ago" in text:
        days = int(text.split()[0])
        return today - timedelta(days=days)
    else:
        return pd.to_datetime(text)

df["Standard_Date"] = df["Review_Date"].apply(convert_date)
print(df)

       Review_Date              Standard_Date
0        Yesterday 2026-03-25 09:21:56.739515
1       2 days ago 2026-03-24 09:21:56.739539
2  March 5th, 2023 2023-03-05 00:00:00.000000


```
3. Why are relative dates problematic?
Ans:
Not in standard format
Cannot sort or compare easily
Difficult for time-based analysis

Section B: Data Wrangling

1. Multi-table Integration

In [ ]:
#sample dataFrames
df_students = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Name": ["A", "B", "C"],
    "Course_ID": [101, 102, 101]
})

df_courses = pd.DataFrame({
    "Course_ID": [101, 102],
    "Course_Name": ["ML", "DS"],
    "Instructor": ["X", "Y"]
})

df_scores = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Score": [80, 90, 70]
})

#merge tables
df = df_students.merge(df_courses, on="Course_ID", how="inner")
df = df.merge(df_scores, on="Student_ID", how="inner")

print(df)

   Student_ID Name  Course_ID Course_Name Instructor  Score
0           1    A        101          ML          X     80
1           2    B        102          DS          Y     90
2           3    C        101          ML          X     70


```
1. Explain how you would combine all three DataFrames to link: • students • their courses • their scores along with the diffeneces of each join.
Ans:
Merge DF_Students + DF_Courses:
Join on Course_ID

Merge result + DF_Scores:
Join on Student_ID

Use pandas merge()

Inner join:
Keeps only matching rows in all tables

Left join:
Keeps all students
Missing course/score → NaN

2. Grouping & Aggregation

In [ ]:
#average score per course
avg_scores = df.groupby("Course_Name")["Score"].mean().reset_index()
print(avg_scores)

  Course_Name  Score
0          DS   90.0
1          ML   75.0


```
2. Grouping & Aggregation?
Ans:
Calculate average score:
Use groupby("Course_Name") on merged data
Apply mean() on Score column

Groupby operation:
Groups data based on a column (Course_Name)
Performs aggregation (mean, sum, count) on each group

3. Data Alignment & Update

In [ ]:
df_scores_update = pd.DataFrame({
    "Student_ID": [2],
    "Score": [95]
})

#set index for alignment
df_scores.set_index("Student_ID", inplace=True)
df_scores_update.set_index("Student_ID", inplace=True)

#update values
df_scores.update(df_scores_update)

#reset index
df_scores.reset_index(inplace=True)

print(df_scores)

   Student_ID  Score
0           1     80
1           2     95
2           3     70


```
3. Data Alignment & Update?
Ans:
Update without duplicates:
Set Student_ID as index in both DataFrames
Use update() to modify existing values

Function used:
pandas update()

Aligns data based on keys (Student_ID)
Updates only matching rows